# Lab W7: Klasifikasi Teks - IndoNLU SmSA

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `07_W7_Text_Transformers_Repo_Adoption.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W7 sudah dibaca, terutama §1.1 (kenapa contextual embeddings), §1.2 (tokenization primer + 3 gaya), §1.3 (frozen vs fine-tune contoh waktu/biaya, [CLS] vs mean pool pemahaman).
- Bab W3 §2.4 sudah dibaca (taxonomy engineered/extracted/learned).
- Familiar dengan sklearn (TF-IDF, LogisticRegression).

**Yang akan Anda hasilkan di akhir lab:**
- 3 representasi teks dilatih dan dibandingkan: TF-IDF+LR (engineered), LSA Embedding+MLP (extracted), IndoBERT fine-tune (learned).
- Tabel perbandingan accuracy + waktu training (gunakan sebagai data konkret untuk §1.3 W7).
- Confusion matrix + error analysis 10 contoh.
- 5 pertanyaan refleksi termasuk inductive bias dan keputusan praktis tanpa GPU.

**Resource:**
- **Hardware:** 
  - Representasi 1 dan 2: CPU cukup (~2-5 menit total).
  - Representasi 3 (IndoBERT fine-tune): **GPU sangat dianjurkan**. Di GPU T4 ~5-10 menit; di CPU ~20-30 menit untuk 2 epoch. Kalau GPU tidak tersedia, lewati Representasi 3 dan kerjakan sebagai bonus saat akses GPU tersedia (Colab Free Tier menyediakan T4).
- **Estimasi waktu kerja:** 90-120 menit (Rep 1+2) atau 2-3 jam dengan Rep 3.

**Pendamping:** Bab W7 di `07_W7_Text_Transformers_Repo_Adoption.md`.

**Referensi bab:** Bab 07 (Section 1) + Bab 03 Section 2.4  
**Estimasi waktu:** 90-120 menit (Representasi 1-2 di CPU, Representasi 3 opsional di GPU/Colab)  
**Dataset:** IndoNLU SmSA (SMS Sentiment Analysis Bahasa Indonesia)

---

## Tujuan Lab

Semua lab sebelumnya bekerja pada domain gambar. Lab ini memperkenalkan domain teks agar Anda tidak bergantung pada intuisi visual saja. Tujuan spesifik:

1. Membandingkan tiga strategi representasi teks (engineered / extracted / learned) - padanan teks dari Lab W6 feature representation.
2. Mengidentifikasi perbedaan *inductive bias* antara pipeline teks dan gambar.
3. Menghubungkan hasil ke taksonomi representasi di Bab W3 Section 2.4.

---

## Setup

In [ ]:
# ── Install dependencies (jalankan sekali) ───────────────────────────────────
# datasets dan transformers tidak tersedia by default di Google Colab
!pip install -q datasets transformers

import time
import numpy as np
import pandas as pd
from datasets import load_dataset

# Kunci seed untuk reproducibility
SEED = 42
np.random.seed(SEED)

print("Setup OK")

## 1. Load Dataset SmSA

**IndoNLU SmSA** - SMS Sentiment Analysis Bahasa Indonesia. Label: positif (0), negatif (1), netral (2).  
Sumber: `indonlp/indonlu`, subset `smsa`.

In [ ]:
ds = load_dataset("indonlp/indonlu", "smsa")
print("Dataset splits:", ds)

train_data = ds["train"]
val_data   = ds["validation"]
test_data  = ds["test"]

print(f"\nTrain: {len(train_data):,} sampel")
print(f"Val:   {len(val_data):,} sampel")
print(f"Test:  {len(test_data):,} sampel")

# Lihat beberapa contoh
for i in range(3):
    print(f"\n[{i}] text: {train_data[i]['text'][:80]}...")
    print(f"    label: {train_data[i]['label']}")

In [ ]:
import matplotlib.pyplot as plt

# Distribusi label
label_names = ["positif", "negatif", "netral"]
train_labels = train_data["label"]

counts = [train_labels.count(i) for i in range(3)]
plt.figure(figsize=(6, 3))
plt.bar(label_names, counts, color=["steelblue", "salmon", "gray"])
plt.title("Distribusi Label SmSA (Train)")
plt.ylabel("Jumlah sampel")
plt.tight_layout()
plt.show()

print("Distribusi:", dict(zip(label_names, counts)))

---

## Representasi 1: Engineered - TF-IDF + Logistic Regression

Representasi *engineered*: fitur dibuat dengan aturan yang dirancang manusia (TF-IDF = frekuensi term yang dinormalisasi oleh keunikannya di corpus). Tidak ada parameter yang dipelajari dari data di tahap representasi ini.

**Hubungan ke Lab 1b**: ini adalah padanan teks dari HOG+MLP.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train = [row["text"] for row in train_data]
y_train = [row["label"] for row in train_data]
X_val   = [row["text"] for row in val_data]
y_val   = [row["label"] for row in val_data]
X_test  = [row["text"] for row in test_data]
y_test  = [row["label"] for row in test_data]

t0 = time.time()

# Ekstraksi fitur TF-IDF
tfidf = TfidfVectorizer(
    max_features=20_000,
    ngram_range=(1, 2),  # unigram + bigram
    min_df=2,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

# Classifier
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr.fit(X_train_tfidf, y_train)

t_rep1 = time.time() - t0

acc_val_r1  = accuracy_score(y_val, lr.predict(X_val_tfidf))
acc_test_r1 = accuracy_score(y_test, lr.predict(X_test_tfidf))

print(f"Representasi 1 - TF-IDF + LR")
print(f"  Val accuracy : {acc_val_r1:.4f}")
print(f"  Test accuracy: {acc_test_r1:.4f}")
print(f"  Waktu training: {t_rep1:.1f}s")
print(f"  Parameter (n_features): {X_train_tfidf.shape[1]:,}")
print()
print(classification_report(y_test, lr.predict(X_test_tfidf), target_names=label_names))

---

## Representasi 2: Extracted - FastText Frozen + MLP

Representasi *extracted*: backbone yang sudah dilatih (FastText) dipakai sebagai feature extractor - bobotnya tidak berubah. Kita rata-ratakan vektor kata untuk mendapat representasi kalimat.

**Hubungan ke Lab 1b**: ini adalah padanan teks dari ResNet-18 frozen → linear probe.

> **Catatan**: Sel ini memerlukan `fasttext` atau `gensim` + model FastText Bahasa Indonesia.  
> Alternatif yang lebih ringan: pakai sentence-level TF-IDF + SVD (LSA) sebagai proxy embedding.

In [ ]:
# Alternatif ringan: TF-IDF + SVD (Latent Semantic Analysis)
# Ini adalah proxy untuk word embedding rata-rata - lebih mudah diinstall
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

t0 = time.time()

# Bangun representasi 128-dim via SVD (mirip "embedding" tanpa pretrained weights)
pipe_r2 = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30_000, ngram_range=(1, 1),
                              sublinear_tf=True, min_df=2)),
    ("svd",   TruncatedSVD(n_components=128, random_state=SEED)),
    ("mlp",   MLPClassifier(hidden_layer_sizes=(64,), max_iter=50,
                            random_state=SEED, early_stopping=True,
                            validation_fraction=0.1)),
])

pipe_r2.fit(X_train, y_train)
t_rep2 = time.time() - t0

acc_val_r2  = accuracy_score(y_val, pipe_r2.predict(X_val))
acc_test_r2 = accuracy_score(y_test, pipe_r2.predict(X_test))

n_params_r2 = sum(w.size for w in pipe_r2.named_steps['mlp'].coefs_)

print(f"Representasi 2 - LSA Embedding + MLP")
print(f"  Val accuracy : {acc_val_r2:.4f}")
print(f"  Test accuracy: {acc_test_r2:.4f}")
print(f"  Waktu training: {t_rep2:.1f}s")
print(f"  Parameter MLP: {n_params_r2:,}")

In [ ]:
# === OPSIONAL: FastText ID yang sesungguhnya ===
# Uncomment jika Anda punya akses ke model FastText Indonesia
#
# import fasttext
# ft_model = fasttext.load_model('cc.id.300.bin')  # download dari fasttext.cc
#
# def get_fasttext_embedding(texts, model):
#     embeddings = []
#     for text in texts:
#         words = text.split()
#         if not words:
#             embeddings.append(np.zeros(model.get_dimension()))
#         else:
#             vecs = [model.get_word_vector(w) for w in words]
#             embeddings.append(np.mean(vecs, axis=0))
#     return np.array(embeddings)
#
# X_train_ft = get_fasttext_embedding(X_train, ft_model)
# X_val_ft   = get_fasttext_embedding(X_val, ft_model)
# X_test_ft  = get_fasttext_embedding(X_test, ft_model)
#
# from sklearn.neural_network import MLPClassifier
# mlp_ft = MLPClassifier(hidden_layer_sizes=(128,), max_iter=50, random_state=SEED)
# mlp_ft.fit(X_train_ft, y_train)
# acc_ft = accuracy_score(y_test, mlp_ft.predict(X_test_ft))
# print(f"FastText + MLP test accuracy: {acc_ft:.4f}")

print("(FastText opsional - lihat komentar di atas)")

---

## Representasi 3: Learned - Fine-tune IndoBERT

Representasi *learned*: backbone (IndoBERT) dan classifier head sama-sama diperbarui selama training. Semua parameter beradaptasi ke tugas.

**Hubungan ke Lab 1b**: ini adalah padanan teks dari ResNet-18 fine-tune penuh.

> **Catatan**: Sel ini paling efisien di GPU (Colab/RunPod). Di CPU, 2 epoch ~20-30 menit.  
> Model: `indobenchmark/indobert-lite-base-p1` (~66M parameter, versi ringan).

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed as hf_set_seed,
)
from torch.utils.data import Dataset

hf_set_seed(SEED)  # seed untuk reproducibility HuggingFace

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

MODEL_NAME = "indobenchmark/indobert-lite-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")

In [ ]:
class SmSADataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        enc = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

print("Membangun dataset (tokenisasi ~1-2 menit)...")
t0 = time.time()
train_dataset = SmSADataset(X_train, y_train, tokenizer)
val_dataset   = SmSADataset(X_val,   y_val,   tokenizer)
test_dataset  = SmSADataset(X_test,  y_test,  tokenizer)
print(f"Tokenisasi selesai dalam {time.time()-t0:.1f}s")

In [ ]:
from sklearn.metrics import accuracy_score as sk_acc

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": sk_acc(labels, preds)}

model_r3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(device)

n_params_r3 = sum(p.numel() for p in model_r3.parameters() if p.requires_grad)
print(f"Total parameter yang dilatih: {n_params_r3:,}")

training_args = TrainingArguments(
    output_dir="/tmp/smsa_indobert",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to="none",
    no_cuda=(device == "cpu"),
)

trainer = Trainer(
    model=model_r3,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

t0 = time.time()
trainer.train()
t_rep3 = time.time() - t0

# Evaluasi pada test set
test_results = trainer.evaluate(test_dataset)
acc_val_r3  = trainer.evaluate(val_dataset)["eval_accuracy"]
acc_test_r3 = test_results["eval_accuracy"]

print(f"\nRepresentasi 3 - IndoBERT fine-tune")
print(f"  Val accuracy : {acc_val_r3:.4f}")
print(f"  Test accuracy: {acc_test_r3:.4f}")
print(f"  Waktu training: {t_rep3:.1f}s")
print(f"  Parameter: {n_params_r3:,}")

---

## Perbandingan Tiga Representasi

In [ ]:
# Isi angka dari sel di atas (atau tulis manual jika Representasi 3 dilewat)
results = {
    "Representasi": [
        "1. TF-IDF + LR (engineered)",
        "2. LSA Embedding + MLP (extracted)",
        "3. IndoBERT fine-tune (learned)",
    ],
    "Val Acc": [acc_val_r1, acc_val_r2, acc_val_r3 if 'acc_val_r3' in dir() else "N/A"],
    "Test Acc": [acc_test_r1, acc_test_r2, acc_test_r3 if 'acc_test_r3' in dir() else "N/A"],
    "Waktu (s)": [t_rep1, t_rep2, t_rep3 if 't_rep3' in dir() else "N/A"],
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
# Bar chart perbandingan
labels_short = ["TF-IDF+LR\n(engineered)", "LSA+MLP\n(extracted)", "IndoBERT\n(learned)"]
accs = [acc_test_r1, acc_test_r2]
if 'acc_test_r3' in dir():
    accs.append(acc_test_r3)
    x_labels = labels_short
else:
    x_labels = labels_short[:2]

colors = ["#4878cf", "#6acc65", "#d65f5f"]
plt.figure(figsize=(7, 4))
bars = plt.bar(x_labels, accs, color=colors[:len(accs)])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{acc:.3f}", ha="center", va="bottom", fontsize=10)
plt.ylim(0, 1.0)
plt.ylabel("Test Accuracy")
plt.title("Perbandingan Representasi Teks - SmSA")
plt.tight_layout()
plt.show()

---

## Analisis Error: Apa yang Sulit?

Identifikasi sampel yang salah diklasifikasi oleh model terbaik Anda.

In [ ]:
# Analisis error dari Representasi 1 (TF-IDF + LR)
y_pred_r1 = lr.predict(X_test_tfidf)
errors = [
    (X_test[i], label_names[y_test[i]], label_names[y_pred_r1[i]])
    for i in range(len(y_test))
    if y_test[i] != y_pred_r1[i]
]

print(f"Total error: {len(errors)} dari {len(y_test)} ({len(errors)/len(y_test)*100:.1f}%)")
print()
print("10 contoh error pertama:")
print("-" * 70)
for text, true, pred in errors[:10]:
    print(f"  Teks     : {text[:60]}...")
    print(f"  Benar    : {true}  |  Prediksi: {pred}")
    print()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred_r1, normalize="true")
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=label_names,
            yticklabels=label_names, cmap="Blues")
plt.xlabel("Prediksi")
plt.ylabel("Label Sebenarnya")
plt.title("Confusion Matrix - TF-IDF + LR (test, normalized)")
plt.tight_layout()
plt.show()

---

## Refleksi

Jawab pertanyaan berikut di sel markdown ini (hapus tanda `> ` dan isi dengan jawaban Anda):

**1. Taksonomi representasi (Bab 01 Section 2.6):**

Petakan ketiga representasi di lab ini ke taksonomi engineered / extracted / learned:
- TF-IDF: `[isi jawaban Anda]`
- LSA Embedding: `[isi jawaban Anda]`
- IndoBERT fine-tune: `[isi jawaban Anda]`

Apakah pemetaan ini sama persis dengan Lab 1b (HOG / ResNet-18 frozen / ResNet-18 fine-tune)? Jika ada perbedaan, jelaskan.

**2. Inductive bias:**

CNN memiliki *spatial inductive bias* - ia mengasumsikan fitur lokal yang berurutan secara spasial. Apa *inductive bias* dari TF-IDF? Apakah TF-IDF "tahu" urutan kata? Bagaimana ini memengaruhi jenis teks yang mudah dan sulit diklasifikasi?

**3. Keputusan praktis:**

Jika Anda punya hanya 500 sampel berlabel dan tidak ada GPU, representasi mana yang Anda pilih dan mengapa? Jawaban tidak harus tunggal - boleh bersyarat.

**4. Hipotesis yang dapat diuji:**

Berdasarkan error analysis, tulis satu hipotesis spesifik tentang mengapa salah satu model membuat error tertentu. Formulasikan sebagai hipotesis yang bisa diuji dengan eksperimen sederhana (sesuai format Bab 02).

**5. Generalisasi ke domain lain:**

Dari pengalaman di Lab 1b dan Lab 5b, apa pola umum yang Anda lihat tentang kapan strategi *extracted* (frozen backbone) cukup vs kapan *learned* (fine-tune) lebih baik? Formulasikan sebagai aturan praktis.

---

## Daftar Periksa Penyelesaian Lab

- [ ] Representasi 1 (TF-IDF + LR) berhasil dijalankan dan menghasilkan akurasi test > 0.70.
- [ ] Representasi 2 (LSA Embedding + MLP) berhasil dijalankan.
- [ ] Tabel perbandingan lengkap (minimal 2 representasi).
- [ ] Confusion matrix ditampilkan.
- [ ] 5 pertanyaan refleksi dijawab.
- [ ] (Bonus) Representasi 3 (IndoBERT fine-tune) dijalankan dan hasilnya ditambahkan ke tabel.